# Lesson 11 練習｜圖片與檔案下載

把每一格的 `None` 或 `# TODO` 補完，再按 Run Cell。
每題最後的 `assert` 是自我檢查：沒有出現錯誤、而且印出「完成」，就代表答對了。

- ⭐ Core：第 1–4 題（圖片是 bytes、存檔、找網址、取檔名）
- ⭐⭐ Practice：第 5–8 題（單張與批次下載、Pillow 讀圖與縮圖）
- ⭐⭐⭐ Challenge：第 9–10 題（整理資料夾、縮圖牆）

這份練習只用本機的練習站（`/gallery` 圖庫）與 `fixtures/gallery` 裡的圖片，不需要外部網路。
生成的檔案都放在 `output/`。


### Notebook 使用方式

先選擇課程 Python 環境，再由上到下執行每一格（Shift + Enter）。先執行工作資料夾設定及前面的匯入。
若看到 `NameError`，先確認變數名稱，並回到上方執行建立該變數的格子；必要時重新啟動核心並從頭執行。練習或作業格要先完成 TODO，再執行檢查。


## 設定工作資料夾

請先執行下一格。使用 VS Code 開啟完整教材資料夾；本堂的 `fixtures/` 和 `output/` 會以本堂資料夾為基準。


In [ ]:
import os
from pathlib import Path

# 從教材根目錄或本堂資料夾啟動 Notebook 都可以。
current = Path.cwd().resolve()
lesson_name = 'lesson11_images_and_files'
candidates = [current, *current.parents]
lesson_path = next((p for p in candidates if p.name == lesson_name and (p / 'fixtures').is_dir()), None)
if lesson_path is None:
    lesson_path = next((p / 'lessons' / lesson_name for p in candidates if (p / 'lessons' / lesson_name / 'fixtures').is_dir()), None)
if lesson_path is None:
    raise FileNotFoundError('找不到本堂教材。請用 VS Code 開啟完整教材資料夾，重新啟動核心後再執行。')
os.chdir(lesson_path)
print('工作資料夾：', Path.cwd())

import sys
tools_path = str(lesson_path.parents[1] / 'tools')
if tools_path not in sys.path:
    sys.path.insert(0, tools_path)


In [ ]:
import io
import re
import shutil
import time
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from PIL import Image, ImageOps

from practice_site import start_server, stop_server

server, BASE_URL = start_server()  # 本機練習站：/gallery 是圖庫，/images/ 放圖片
GALLERY = Path("fixtures/gallery")  # 同一批圖片的本機副本，第 7 題起會用到
print("本機練習站：", BASE_URL)


## ⭐ 第 1 題｜圖片是二進位資料

用 `requests.get` 抓 `f"{BASE_URL}/images/cover_01.jpg"`，結果存成 `response`。再取出：
- `data`：圖片的原始內容（bytes）
- `content_type`：回應標頭裡的 `Content-Type`


In [ ]:
# TODO: 請在這裡寫程式
response = None
data = None
content_type = None


In [ ]:
assert isinstance(data, bytes) and len(data) == 31047
assert content_type == "image/jpeg"
assert data[:2] == b"\xff\xd8", "JPEG 檔案一定是以這兩個位元組開頭"
print("第 1 題完成：", len(data), "bytes", content_type)


## ⭐ 第 2 題｜存成檔案

把第 1 題的 `data` 用「二進位寫入」存到 `saved_path`（資料夾不存在要先建立）。
存好之後，用 `stat().st_size` 取得檔案大小，存成 `saved_size`。


In [ ]:
saved_path = Path("output/q2/cover_01.jpg")

# TODO: 請在這裡寫程式
saved_size = None


In [ ]:
assert saved_path.exists()
assert saved_size == len(data)
with Image.open(saved_path) as picture:
    assert picture.size == (318, 395)
print("第 2 題完成：", saved_path, saved_size, "bytes")


## ⭐ 第 3 題｜找出頁面上所有圖片的網址

抓 `page_url`（圖庫頁），用 BeautifulSoup 找出每一個 `<img>` 真正的圖片網址，
存成 `image_urls`：完整網址的 list，依頁面順序。

提醒：
- `src` 常常是相對路徑，要用 `urljoin(頁面網址, src)` 變成完整網址。
- 有些圖片是「懶載入」：`src` 只是佔位圖（loading.gif），真正的網址放在 `data-src`。


In [ ]:
page_url = f"{BASE_URL}/gallery"

# TODO: 請在這裡寫程式
image_urls = None


In [ ]:
assert len(image_urls) == 17
assert image_urls[0] == f"{BASE_URL}/images/cover_01.jpg"
assert f"{BASE_URL}/images/cover_12.jpg" in image_urls, "第 12 張是懶載入，要看 data-src"
assert not any("loading.gif" in url for url in image_urls)
print("第 3 題完成：共", len(image_urls), "張（含 2 張有問題的）")


## ⭐ 第 4 題｜從網址取出檔名

寫 `filename_from_url(url)`，回傳網址最後的檔名。`?` 後面的參數（例如 `?v=3`）不算檔名的一部分。
提示：`urlparse(url).path` 可以只取網址的路徑部分。


In [ ]:
def filename_from_url(url):
    # TODO: 請在這裡寫程式
    return None


In [ ]:
assert filename_from_url("http://127.0.0.1:8000/images/cover_01.jpg") == "cover_01.jpg"
assert filename_from_url("http://127.0.0.1:8000/images/cover_02.jpg?v=3") == "cover_02.jpg"
assert [filename_from_url(url) for url in image_urls[:3]] == ["cover_01.jpg", "cover_02.jpg", "cover_03.jpg"]
print("第 4 題完成")


## ⭐⭐ 第 5 題｜單張下載函式

寫 `download_image(url, folder)`：
- 用 `requests.get` 抓（`timeout=10`）
- 狀態碼不是 200，或 `Content-Type` 不是以 `image/` 開頭 → 回傳 `None`，**不要存檔**
- 否則存到 `folder / 檔名`（`folder` 不存在要先建立；檔名用第 4 題的函式），回傳存好的路徑（`Path`）

小提醒：有些網站圖片不見了，卻回傳 200 加上一個錯誤網頁。所以光看狀態碼不夠，要看 `Content-Type`。


In [ ]:
def download_image(url, folder):
    # TODO: 請在這裡寫程式
    return None


In [ ]:
q5_folder = Path("output/q5")
shutil.rmtree(q5_folder, ignore_errors=True)  # 每次都從空資料夾開始

good = download_image(f"{BASE_URL}/images/cover_03.jpg", q5_folder)
missing = download_image(f"{BASE_URL}/images/missing.jpg", q5_folder)  # 404
fake = download_image(f"{BASE_URL}/images/fake.jpg", q5_folder)  # 200，但內容是網頁

assert good == q5_folder / "cover_03.jpg" and good.stat().st_size == 9403
assert missing is None, "404 不該存檔"
assert fake is None, "Content-Type 不是 image/ 不該存檔"
assert [path.name for path in q5_folder.iterdir()] == ["cover_03.jpg"]
print("第 5 題完成：", good)


## ⭐⭐ 第 6 題｜批次下載

寫 `download_all(urls, folder, delay=0.0)`：用第 5 題的函式下載每一個網址，回傳統計 dict：

```python
{"ok": 新下載成功幾張, "skipped": 檔案已經存在而略過幾張, "failed": 失敗幾張}
```

- 檔名已經在 `folder` 裡，就略過（不重新下載）
- 每下載一張，之後要 `time.sleep(delay)`（真實網站請設 1 秒以上；練習站不用等，所以預設 0）


In [ ]:
def download_all(urls, folder, delay=0.0):
    # TODO: 請在這裡寫程式
    return None


In [ ]:
q6_folder = Path("output/q6")
shutil.rmtree(q6_folder, ignore_errors=True)

first = download_all(image_urls, q6_folder)
second = download_all(image_urls, q6_folder)  # 再跑一次：全部都該略過

assert first == {"ok": 15, "skipped": 0, "failed": 2}, first
assert second == {"ok": 0, "skipped": 15, "failed": 2}, second
assert len(list(q6_folder.iterdir())) == 15
print("第 6 題完成：", first, "→", second)


## ⭐⭐ 第 7 題｜用 Pillow 看圖片資訊

寫 `image_info(path)`：回傳 `(寬, 高, 格式, 模式)`，例如 `(318, 395, "JPEG", "RGB")`。
提示：`Image.open(path)` 之後有 `.size`、`.format`、`.mode`；用 `with` 開啟，用完會自動關檔。
這一題與下一題都直接用 `GALLERY`（`fixtures/gallery`）裡的圖片。


In [ ]:
def image_info(path):
    # TODO: 請在這裡寫程式
    return None


In [ ]:
assert image_info(GALLERY / "cover_01.jpg") == (318, 395, "JPEG", "RGB")
assert image_info(GALLERY / "banner.png") == (1200, 400, "PNG", "RGBA")
assert image_info(GALLERY / "icon.png") == (64, 64, "PNG", "RGBA")
print("第 7 題完成")


## ⭐⭐ 第 8 題｜做縮圖

寫 `make_thumb(src, dst, size=(120, 160))`：
- 開啟 `src`，用 `ImageOps.fit(圖片, size)` 裁切並縮成「剛好」`size` 的大小
- 如果圖片是 RGBA（有透明），先貼到白色背景上變成 RGB，否則不能存成 JPEG
- 存成 JPEG（`quality=85`），回傳 `dst`

提示：貼到白色背景：`background = Image.new("RGB", 圖.size, "white")`，再 `background.paste(圖, mask=圖.getchannel("A"))`。


In [ ]:
def make_thumb(src, dst, size=(120, 160)):
    # TODO: 請在這裡寫程式
    return None


In [ ]:
q8_folder = Path("output/q8")
shutil.rmtree(q8_folder, ignore_errors=True)

for name in ["cover_01.jpg", "landscape.jpg", "banner.png", "icon.png"]:
    dst = q8_folder / f"{Path(name).stem}_thumb.jpg"
    returned = make_thumb(GALLERY / name, dst)
    assert Path(returned) == dst, "要回傳 dst"
    assert image_info(dst) == (120, 160, "JPEG", "RGB"), (name, image_info(dst))

with Image.open(q8_folder / "icon_thumb.jpg") as icon_thumb:
    assert min(icon_thumb.getpixel((0, 0))) > 240, "透明的地方要變成白色，不是黑色"
print("第 8 題完成")


## ⭐⭐⭐ 第 9 題｜依評分整理資料夾

下一格已經寫好 `books`：圖庫裡 12 本書的 `file`（檔名）、`title`（書名）、`rating`（評分，`One` 到 `Five`）。
請完成兩個函式：

1. `slugify(title)`：書名轉成適合當檔名的字串。
   轉小寫 → 連續的「非英文字母、非數字」字元換成一個 `-` → 去掉頭尾的 `-` → 最多取 40 個字元 → 再去掉尾端的 `-`。
2. `organize_by_rating(books, source_folder, target_folder)`：
   對每一本書（編號從 1 開始），用第 8 題的 `make_thumb` 做縮圖，存到
   `target_folder / 評分 / f"{編號:02d}_{slugify(書名)}.jpg"`，來源是 `source_folder / 檔名`。
   回傳 dict：評分 → 該評分底下的檔名 list（依編號）。


In [ ]:
def books_info(page_url):
    """已經寫好：從圖庫頁整理出 12 本書的資料。"""
    soup = BeautifulSoup(requests.get(page_url, timeout=10).text, "html.parser")
    result = []
    for figure in soup.select("figure.book"):
        img = figure.select_one("img")
        real_url = urljoin(page_url, img.get("data-src") or img["src"])
        result.append(
            {
                "file": Path(urlparse(real_url).path).name,
                "title": figure.select_one("figcaption").get_text(),
                "rating": figure["data-rating"],
            }
        )
    return result


books = books_info(f"{BASE_URL}/gallery")
print(len(books), books[0])


def slugify(title):
    # TODO: 請在這裡寫程式
    return None


def organize_by_rating(books, source_folder, target_folder):
    # TODO: 請在這裡寫程式
    return None


In [ ]:
assert len(books) == 12
assert slugify("Sapiens: A Brief History of Humankind") == "sapiens-a-brief-history-of-humankind"
assert slugify("Starving Hearts (Triangular Trade Trilogy, #1)") == "starving-hearts-triangular-trade-trilogy"

q9_folder = Path("output/q9")
shutil.rmtree(q9_folder, ignore_errors=True)
organized = organize_by_rating(books, GALLERY, q9_folder)

assert {rating: len(names) for rating, names in organized.items()} == {"One": 4, "Two": 1, "Three": 2, "Four": 4, "Five": 1}
assert organized["Five"] == ["05_sapiens-a-brief-history-of-humankind.jpg"]
assert organized["Two"] == ["11_starving-hearts-triangular-trade-trilogy.jpg"]
assert (q9_folder / "Four" / "07_the-dirty-little-secrets-of-getting-your.jpg").exists()
assert sorted(path.name for path in q9_folder.iterdir()) == ["Five", "Four", "One", "Three", "Two"]
for path in q9_folder.rglob("*.jpg"):
    assert image_info(path)[:2] == (120, 160)
print("第 9 題完成：", {rating: len(names) for rating, names in organized.items()})


## ⭐⭐⭐ 第 10 題｜縮圖牆

寫 `contact_sheet(paths, cols=4, cell=(120, 160), gap=8)`：把很多張圖排成一張大圖（Pillow 的 `Image` 物件）並回傳。

- 每張圖用 `ImageOps.fit` 縮成 `cell` 的大小，由左到右、由上到下排列
- 格子之間與四周都留 `gap` 像素的白色空隙
- 大圖的寬 = `cols × 格寬 + (cols + 1) × gap`；高度同理（列數用「圖片數 ÷ cols」無條件進位）

提示：`Image.new("RGB", (寬, 高), "white")` 建立白色底圖，`sheet.paste(圖, (x, y))` 貼上一張圖。


In [ ]:
def contact_sheet(paths, cols=4, cell=(120, 160), gap=8):
    # TODO: 請在這裡寫程式
    return None


In [ ]:
cover_paths = [GALLERY / book["file"] for book in books]
wall = contact_sheet(cover_paths)
Path("output").mkdir(exist_ok=True)
wall.save("output/q10_wall.jpg")

assert wall.size == (4 * 120 + 5 * 8, 3 * 160 + 4 * 8)  # (520, 512)
assert wall.getpixel((0, 0)) == (255, 255, 255), "四周的空隙是白色"
assert sum(wall.getpixel((8 + 120 + 8 + 60, 8 + 80))) < 300, "第 2 張（深色封面）的中間不該是白色"
print("第 10 題完成：", wall.size, "→ 打開 output/q10_wall.jpg 看看")


## 收尾：關閉練習站


In [ ]:
stop_server(server)
print("練習站已關閉")
